# EDA — US License Plates (phân loại biển số xe theo bang của Mỹ)

**Bài toán:** nhìn ảnh một biển số và cho biết biển đó thuộc bang / vùng lãnh thổ nào của Mỹ (56 lớp: 50 bang, Washington DC và 5 vùng lãnh thổ).

**Dữ liệu:** bộ *US license plates – Image Classification*, đã có sẵn trong thư mục `data/new_plates(raw_dataset)`.

| Phần | Nội dung | Dùng để quyết định |
|---|---|---|
| 1 | Tổng quan: thư mục, số ảnh, danh sách lớp, **kiểm tra tên lớp và file `plates.csv`** | Có cần sửa nhãn không |
| 2 | Quét toàn bộ ảnh (một lượt) | Nguồn số liệu cho các phần sau |
| 3 | Phân phối lớp | Có cần xử lý mất cân bằng, có cần chia lại không |
| 4 | Kích thước, định dạng, ảnh hỏng | Bước clean, kích thước resize |
| 5 | Ảnh mẫu | Hiểu dữ liệu, chọn augment hợp lý |
| 6 | Thống kê pixel theo kênh màu | Cách normalize |
| 7 | Độ sáng, tương phản theo lớp | Mức color jitter |
| 8 | Ảnh trung bình mỗi lớp | Có nên lật / xoay / crop không |
| 9 | Độ nét, mẫu bất thường | Bước clean |
| 10 | **Ảnh trùng lặp** | Bước clean, cách chia dữ liệu |
| 11 | Độ giống nhau giữa các lớp | Dự đoán cặp lớp dễ nhầm |
| 12 | Tổng hợp và kết luận | Danh sách việc cho bước tiền xử lý |

**Đầu ra:** mọi biểu đồ và số liệu được lưu vào `data/eda_outputs/`. Bước tiền xử lý dùng `data/eda_outputs/duplicate_pairs.csv` (danh sách cặp ảnh giống nhau).

In [ ]:
import io, os, json, hashlib, random, difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image as PILImage
from tqdm.auto import tqdm
from IPython.display import display, Image as IPyImage

SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATA_DIR = "data/new_plates(raw_dataset)"   # thư mục dữ liệu; sửa lại nếu bạn để chỗ khác
OUT = "data/eda_outputs"            # mọi biểu đồ và số liệu được lưu vào đây
os.makedirs(OUT, exist_ok=True)

# ---- kiểu biểu đồ dùng chung
C = dict(surface="#fcfcfb", text="#0b0b0b", text2="#52514e", grid="#e6e5e0",
         blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", red="#e34948", green="#008300")
SEQ = LinearSegmentedColormap.from_list(
    "seq_blue", ["#f0efec", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": C["surface"], "axes.facecolor": C["surface"], "savefig.facecolor": C["surface"],
    "figure.dpi": 100, "savefig.dpi": 120, "savefig.bbox": "tight",
    "font.size": 10, "text.color": C["text"],
    "axes.edgecolor": C["grid"], "axes.labelcolor": C["text2"], "axes.titlecolor": C["text"],
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": C["text2"], "ytick.color": C["text2"], "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
})

RESULTS = {"dataset": "US license plates - Image Classification"}    # số liệu chính, cuối notebook ghi ra JSON

def save(fig, name, photo=False):
    # biểu đồ lưu PNG; lưới ảnh chụp lưu JPEG để file không quá nặng
    if photo:
        path = os.path.join(OUT, name + ".jpg")
        fig.savefig(path, dpi=110, pil_kwargs={"quality": 85})
        plt.close(fig)
        display(IPyImage(filename=path))
    else:
        fig.savefig(os.path.join(OUT, name + ".png"))
        plt.show()

## 1. Đọc dữ liệu và tổng quan

Đọc dữ liệu từ thư mục `DATA_DIR`, dò các thư mục `train` / `valid` / `test`; mỗi thư mục con bên trong là một lớp.

Ảnh của cả ba thư mục được đưa vào **một bảng chung**. Cột `split` chỉ ghi lại ảnh đến từ thư mục nào của bộ gốc, để tham khảo.

In [ ]:
ROOT = DATA_DIR
assert os.path.isdir(ROOT), f"Không thấy thư mục dữ liệu '{ROOT}' — sửa DATA_DIR ở ô cấu hình"
print("Thư mục dữ liệu:", ROOT)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ALIAS = {"train": "train", "training": "train", "valid": "valid", "val": "valid", "validation": "valid",
         "test": "test", "testing": "test"}

split_dirs, csv_files = {}, []
for dp, dns, fns in os.walk(ROOT):
    if os.path.basename(OUT) in dns:                  # bỏ qua thư mục kết quả của chính notebook này
        dns.remove(os.path.basename(OUT))
    depth = dp[len(ROOT):].count(os.sep)
    csv_files += [os.path.join(dp, f) for f in fns if f.lower().endswith(".csv")]
    for d in list(dns):
        key = ALIAS.get(d.lower())
        if key and key not in split_dirs:
            split_dirs[key] = os.path.join(dp, d)
    if depth >= 2:
        dns[:] = []
assert split_dirs, f"Không tìm thấy thư mục train/valid/test trong {ROOT} — kiểm tra lại DATA_DIR"
print("Các split tìm thấy:", split_dirs)
print("File CSV đi kèm:", [os.path.relpath(p, ROOT) for p in csv_files] or "không có")
for p in csv_files[:2]:
    try:
        t = pd.read_csv(p)
        print(f"\n{os.path.basename(p)}: {len(t):,} dòng, cột = {list(t.columns)}")
        display(t.head(3))
    except Exception as e:
        print("Không đọc được", p, repr(e)[:100])

In [ ]:
recs, other_files = [], []
for s in ["train", "valid", "test"]:
    if s not in split_dirs:
        continue
    for cls in sorted(os.listdir(split_dirs[s])):
        cdir = os.path.join(split_dirs[s], cls)
        if not os.path.isdir(cdir):
            continue
        for fn in sorted(os.listdir(cdir)):
            p = os.path.join(cdir, fn)
            if os.path.splitext(fn)[1].lower() in IMG_EXT:
                recs.append((s, cls, p))
            else:
                other_files.append(os.path.relpath(p, ROOT))

index = pd.DataFrame(recs, columns=["split", "label", "path"])
index["idx"] = index.groupby("split").cumcount()
SPLITS = [s for s in ["train", "valid", "test"] if s in set(index["split"])]
CLASS_NAMES = sorted(index["label"].unique())
N_CLS = len(CLASS_NAMES)
CLS_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
PATHS = {s: index[index["split"] == s]["path"].tolist() for s in SPLITS}

overview = pd.DataFrame({"số ảnh": {s: len(PATHS[s]) for s in SPLITS}})
overview["tỉ lệ (%)"] = (overview["số ảnh"] / overview["số ảnh"].sum() * 100).round(1)
overview["số lớp có ảnh"] = [index[index["split"] == s]["label"].nunique() for s in SPLITS]
overview.loc["tổng (bộ đã gộp)"] = [overview["số ảnh"].sum(), 100.0, N_CLS]
overview.index.name = "thư mục gốc"
display(overview)

print(f"Số lớp: {N_CLS}")
print(CLASS_NAMES)
missing = {s: sorted(set(CLASS_NAMES) - set(index[index["split"] == s]["label"])) for s in SPLITS}
print("Lớp vắng mặt ở từng split:", {s: v for s, v in missing.items() if v} or "không có")
print(f"File không phải ảnh nằm trong thư mục lớp: {len(other_files)}", other_files[:10])

RESULTS["splits"] = {s: len(PATHS[s]) for s in SPLITS}
RESULTS["n_classes"] = N_CLS
RESULTS["class_names"] = CLASS_NAMES
RESULTS["classes_missing_per_split"] = missing
RESULTS["csv_files"] = [os.path.relpath(p, ROOT) for p in csv_files]
RESULTS["n_non_image_files"] = len(other_files)

### Kiểm tra tên lớp

So tên từng thư mục lớp với danh sách tên chuẩn của 50 bang, Washington DC và 5 vùng lãnh thổ. Tên nào không khớp thì tìm tên chuẩn gần nhất để gợi ý.

In [ ]:
STATES = '''ALABAMA, ALASKA, ARIZONA, ARKANSAS, CALIFORNIA, COLORADO, CONNECTICUT, DELAWARE, FLORIDA, GEORGIA, HAWAII,
IDAHO, ILLINOIS, INDIANA, IOWA, KANSAS, KENTUCKY, LOUISIANA, MAINE, MARYLAND, MASSACHUSETTS, MICHIGAN, MINNESOTA,
MISSISSIPPI, MISSOURI, MONTANA, NEBRASKA, NEVADA, NEW HAMPSHIRE, NEW JERSEY, NEW MEXICO, NEW YORK, NORTH CAROLINA,
NORTH DAKOTA, OHIO, OKLAHOMA, OREGON, PENNSYLVANIA, RHODE ISLAND, SOUTH CAROLINA, SOUTH DAKOTA, TENNESSEE, TEXAS, UTAH,
VERMONT, VIRGINIA, WASHINGTON, WEST VIRGINIA, WISCONSIN, WYOMING, WASHINGTON DC, AMERICAN SAMOA, GUAM, PUERTO RICO,
NORTHERN MARIANA ISLANDS, US VIRGIN ISLANDS'''
EXPECTED = [x.strip() for x in STATES.replace("\n", " ").split(",")]
assert len(EXPECTED) == 56

odd = []
for c in CLASS_NAMES:
    if c.upper() not in EXPECTED:
        close = difflib.get_close_matches(c.upper(), EXPECTED, n=1, cutoff=0.6)
        odd.append({"tên thư mục": c, "tên chuẩn gần nhất": close[0] if close else "(không có tên gần giống)",
                    "số ảnh": int((index["label"] == c).sum())})
print(f"Tên lớp khớp danh sách chuẩn: {N_CLS - len(odd)} / {N_CLS}")
name_check = pd.DataFrame(odd)
display(name_check)
print("Tên chuẩn không có thư mục tương ứng:", [e for e in EXPECTED if e not in [c.upper() for c in CLASS_NAMES]])
RESULTS["odd_class_names"] = odd

### Kiểm tra file `plates.csv`

Bộ gốc kèm file `plates.csv` liệt kê từng ảnh với bốn cột: `class id`, `filepaths`, `labels`, `data set`. Nếu sau này đọc dữ liệu bằng file này thì nhãn lấy từ cột `labels`, nên cần kiểm tra hai điều:

1. CSV có khớp với ảnh trên đĩa không (đủ dòng, đường dẫn tồn tại, nhãn trùng tên thư mục).
2. Các tên lớp viết sai ở trên có xuất hiện trong CSV không, và ở những cột nào.

In [ ]:
csv_path = next((p for p in csv_files if os.path.basename(p).lower() == "plates.csv"), csv_files[0] if csv_files else None)
need = {"class id", "filepaths", "labels", "data set"}
meta = pd.read_csv(csv_path) if csv_path else None
if meta is None or not need <= set(meta.columns):
    print("Không có plates.csv với đủ 4 cột", sorted(need), "→ bỏ qua phần kiểm tra này")
    RESULTS["csv_check"] = None
else:
    base = os.path.dirname(csv_path)
    parts = meta["filepaths"].str.split("/")
    meta["thư mục split"] = parts.str[0]
    meta["thư mục lớp"] = parts.str[-2]
    exists = [os.path.exists(os.path.join(base, *p.split("/"))) for p in meta["filepaths"]]
    on_disk = {os.path.relpath(p, base).replace(os.sep, "/") for p in index["path"]}
    csv_check = pd.Series({
        "số dòng trong CSV": len(meta),
        "số ảnh trên đĩa": len(index),
        "dòng CSV trỏ tới file không tồn tại": int(len(meta) - sum(exists)),
        "ảnh trên đĩa không có trong CSV": len(on_disk - set(meta["filepaths"])),
        "dòng có cột labels khác tên thư mục lớp": int((meta["labels"] != meta["thư mục lớp"]).sum()),
        "dòng có cột data set khác thư mục split": int((meta["data set"] != meta["thư mục split"]).sum()),
        "số nhãn khác nhau trong cột labels": int(meta["labels"].nunique()),
        "class id ứng với nhiều hơn một nhãn": int((meta.groupby("class id")["labels"].nunique() > 1).sum()),
    }, name="kết quả")
    display(csv_check.to_frame())

    # các tên lớp không chuẩn xuất hiện ở đâu
    where = []
    for o in odd:
        bad, good = o["tên thư mục"], o["tên chuẩn gần nhất"]
        where.append({
            "tên trong bộ gốc": bad, "tên chuẩn": good,
            "thư mục mang tên này": ", ".join(f"{sp} ({int(((index['split'] == sp) & (index['label'] == bad)).sum())})"
                                              for sp in SPLITS if ((index["split"] == sp) & (index["label"] == bad)).any()),
            "dòng CSV: cột labels": int((meta["labels"] == bad).sum()),
            "dòng CSV: cột filepaths": int((meta["thư mục lớp"] == bad).sum()),
            "class id": ", ".join(str(v) for v in sorted(meta.loc[meta["labels"] == bad, "class id"].unique())),
            "dòng CSV dùng tên chuẩn": int((meta["labels"] == good).sum() + (meta["thư mục lớp"] == good).sum()),
        })
    name_in_csv = pd.DataFrame(where)
    display(name_in_csv)
    RESULTS["csv_check"] = csv_check.to_dict()
    RESULTS["odd_names_in_csv"] = where

> **Nhận xét**
>
> - Bộ đã gộp có **8.721 ảnh, 56 lớp**. Bộ gốc chia rất lệch: `train` 8.161 ảnh (93,6%), `valid` 280 (3,2%), `test` 280 (3,2%). Cả 56 lớp đều có mặt ở cả ba thư mục, và trong các thư mục lớp không có file nào không phải ảnh.
> - **Có 4 tên lớp không khớp tên chuẩn**, trong đó hai tên là lỗi chính tả:
>   - `HAWAI` (154 ảnh), đúng phải là `HAWAII`;
>   - `MISSIPPI` (158 ảnh), đúng phải là `MISSISSIPPI`.
>
>   Hai tên còn lại là cách đặt tên của bộ gốc, không phải lỗi: `CNMI` là viết tắt của Commonwealth of the Northern Mariana Islands, còn `U S VIRGIN ISLANDS` chỉ khác tên chuẩn ở một dấu cách.
> - **Lỗi chính tả có ở cả thư mục lẫn file `plates.csv`.** `HAWAI` và `MISSIPPI` là tên thư mục ở cả `train`, `valid` và `test`, đồng thời xuất hiện trong cột `labels` (154 và 158 dòng) và trong đường dẫn ở cột `filepaths` (154 và 158 dòng). Không có dòng nào trong CSV dùng tên viết đúng.
> - Ngoài lỗi tên, `plates.csv` **khớp hoàn toàn với ảnh trên đĩa**: 8.721 dòng cho 8.721 ảnh, không dòng nào trỏ tới file không tồn tại, cột `labels` luôn trùng tên thư mục, và mỗi `class id` chỉ ứng với một nhãn.
>
> **Kết luận:** đây là lỗi **tên lớp**, không phải ảnh bị gán nhầm bang, nên không ảnh hưởng tới việc huấn luyện; nó chỉ làm sai tên hiển thị trong biểu đồ, confusion matrix và báo cáo. EDA không sửa. Ở bước tiền xử lý cần sửa **đồng thời ba chỗ** (tên thư mục, cột `labels`, cột `filepaths`), nếu không CSV sẽ trỏ tới đường dẫn không tồn tại. `class id` của hai lớp (13 và 26) không đổi vì thứ tự bảng chữ cái giữ nguyên.

## 2. Quét toàn bộ ảnh

Đọc từng ảnh đúng một lần và ghi lại mọi thứ các phần sau cần: kích thước, định dạng, giá trị trung bình từng kênh màu, độ sáng, độ tương phản, độ nét, mã băm MD5 và ảnh thu nhỏ để tìm ảnh trùng. Ảnh hỏng được ghi nhận chứ không làm dừng vòng lặp.

Mọi thống kê trong notebook này tính trên **toàn bộ ảnh của bộ đã gộp**, vì ở giai đoạn EDA chưa có cách chia train / valid / test chính thức. Riêng mean / std dùng cho `Normalize` sẽ được tính lại **chỉ trên train** sau khi chia ở bước tiền xử lý.

In [ ]:
MW, MH = 112, 64                                    # kích thước ảnh thu nhỏ dùng cho ảnh trung bình

def load_img(split, idx):
    return PILImage.open(PATHS[split][int(idx)]).convert("RGB")

rows, bad, thumbs = [], [], []
ch_sum, ch_sq, n_px = np.zeros(3), np.zeros(3), 0   # cho mean/std (toàn bộ ảnh)
hist256 = np.zeros((N_CLS, 3, 256))                 # histogram màu theo lớp (toàn bộ ảnh)
mean_img = np.zeros((N_CLS, MH, MW, 3))             # ảnh trung bình theo lớp (toàn bộ ảnh)
mean_cnt = np.zeros(N_CLS)

for r in tqdm(index.itertuples(), total=len(index)):
    y = CLS_ID[r.label]
    try:
        with open(r.path, "rb") as f:
            raw = f.read()
        im = PILImage.open(io.BytesIO(raw)); im.load()
        rgb = im.convert("RGB")
        a = np.asarray(rgb)
        gray = np.asarray(rgb.convert("L"), dtype=np.float32)
    except Exception as e:
        bad.append(dict(split=r.split, idx=r.idx, label=r.label, file=os.path.relpath(r.path, ROOT),
                        error=repr(e)[:150]))
        continue

    # độ nét: phương sai của Laplacian (càng thấp càng mờ)
    if min(gray.shape) >= 3:
        lap = (-4 * gray[1:-1, 1:-1] + gray[:-2, 1:-1] + gray[2:, 1:-1]
               + gray[1:-1, :-2] + gray[1:-1, 2:])
        sharp = float(lap.var())
    else:
        sharp = 0.0
    # ảnh thu nhỏ 28x16 (mức xám, đã chuẩn hoá) dùng để tìm ảnh trùng ở phần 10
    t = np.asarray(rgb.convert("L").resize((28, 16), PILImage.BILINEAR), dtype=np.float32).ravel()
    t = t - t.mean()
    thumbs.append(t / (np.linalg.norm(t) + 1e-6))
    m = a.reshape(-1, 3).mean(0)

    rows.append(dict(
        split=r.split, idx=r.idx, label_id=y, label=r.label, file=os.path.relpath(r.path, ROOT),
        width=im.width, height=im.height, mode=im.mode, format=im.format, kb=len(raw) / 1024,
        mean_r=m[0], mean_g=m[1], mean_b=m[2],
        brightness=float(gray.mean()), contrast=float(gray.std()), sharpness=sharp,
        md5=hashlib.md5(a.tobytes() + str(a.shape).encode()).hexdigest()))

    flat = a.reshape(-1, 3).astype(np.float64) / 255.0
    ch_sum += flat.sum(0); ch_sq += (flat ** 2).sum(0); n_px += flat.shape[0]
    for c in range(3):
        hist256[y, c] += np.bincount(a[..., c].ravel(), minlength=256)
    mean_img[y] += np.asarray(rgb.resize((MW, MH), PILImage.BILINEAR), dtype=np.float64)
    mean_cnt[y] += 1

df = pd.DataFrame(rows)
df["key"] = df["split"] + "/" + df["label"] + "/" + df["file"].map(os.path.basename)   # định danh ảnh trong bộ gốc
df.to_csv(os.path.join(OUT, "per_image_stats.csv"), index=False)
ASPECT = float((df["height"] / df["width"]).median())      # cao / rộng, dùng để vẽ lưới ảnh
print(f"Đọc được {len(df):,} ảnh · lỗi {len(bad)} ảnh")
df.head()

## 3. Phân phối lớp

Đếm số ảnh mỗi lớp trong bộ đã gộp, kèm theo ảnh đó đến từ thư mục nào của bộ gốc. Hai điều cần xem:

- **Các lớp có lệch nhau không.** Nếu lệch nhiều thì dùng class weight hoặc sampler, và báo cáo F1 macro thay vì chỉ accuracy.
- **Cách chia của bộ gốc có dùng được không.** Nếu valid và test chỉ có vài ảnh mỗi lớp thì accuracy đo trên đó dao động mạnh, và nên chia lại ở bước tiền xử lý.

In [ ]:
counts = (df.groupby(["label", "split"]).size().unstack(fill_value=0)
            .reindex(index=CLASS_NAMES, columns=SPLITS, fill_value=0))
counts["tổng"] = counts.sum(axis=1)

summary_cls = pd.DataFrame({
    s: {"ít nhất": counts[s].min(), "nhiều nhất": counts[s].max(),
        "trung bình": round(counts[s].mean(), 1), "trung vị": counts[s].median(),
        "độ lệch chuẩn": round(counts[s].std(), 2),
        "tỉ lệ max/min": round(counts[s].max() / max(counts[s].min(), 1), 3)}
    for s in SPLITS + ["tổng"]})
display(summary_cls)

by_total = counts.sort_values("tổng")
print("5 lớp ít ảnh nhất:");  display(by_total.head(5))
print("5 lớp nhiều ảnh nhất:"); display(by_total.tail(5))

RESULTS["class_counts"] = counts.to_dict(orient="index")
RESULTS["class_balance"] = summary_cls.to_dict()

In [ ]:
order = counts.sort_values("tổng").index
palette = [C["blue"], C["orange"], C["aqua"]]

fig, ax = plt.subplots(figsize=(8.5, 0.24 * N_CLS + 1.6))
left = np.zeros(len(order))
for s, col in zip(SPLITS, palette):
    vals = counts.loc[order, s].values
    ax.barh(order, vals, left=left, height=0.62, color=col, edgecolor=C["surface"], linewidth=1.5, label=s)
    left += vals
for y, v in enumerate(left):
    ax.text(v + left.max() * 0.01, y, f"{int(v):,}", va="center", fontsize=8, color=C["text2"])
ax.set_xlim(0, left.max() * 1.08)
ax.set_ylim(-0.7, len(order) - 0.3)
ax.grid(axis="y", visible=False)
ax.set_xlabel("số ảnh")
ax.set_title("Số ảnh mỗi lớp trong bộ đã gộp (màu: thư mục gốc)", pad=30)
ax.legend(ncol=len(SPLITS), loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0.2, handlelength=1.2)
save(fig, "fig03_class_distribution")

> **Nhận xét**
>
> - **Bộ đã gộp khá cân bằng:** mỗi lớp có 96–185 ảnh, trung bình 156. Nếu không tính hai lớp ít nhất là CNMI (96) và OREGON (123) thì khoảng dao động chỉ còn 141–185.
> - Tỉ lệ lớp nhiều nhất / ít nhất là 1,9, gần như hoàn toàn do CNMI. Mức lệch này nhẹ, **không cần oversampling hay class weight**.
> - **Cách chia của bộ gốc không dùng được để đánh giá:** `valid` và `test` có đúng **5 ảnh mỗi lớp** (các dải cam và xanh lá rất mỏng trong biểu đồ). Với 280 ảnh test, một ảnh đoán sai làm accuracy đổi 0,36 điểm phần trăm và sai số của accuracy tổng vào khoảng ±5,6 điểm (khoảng tin cậy 95%, quanh mức accuracy 65%). Recall của từng lớp chỉ nhận được các giá trị 0, 20, 40, 60, 80, 100%.
>
> **Đề xuất cho bước tiền xử lý:** chia lại bộ đã gộp theo tỉ lệ khoảng 70 / 15 / 15, **stratified theo lớp** và cố định seed. Khi đó mỗi lớp có khoảng 23 ảnh test (lớp ít nhất khoảng 14), tập test có khoảng 1.300 ảnh và sai số của accuracy tổng giảm còn khoảng ±2,6 điểm. Báo cáo accuracy kèm F1 macro; theo dõi riêng CNMI và OREGON.

## 4. Kích thước, định dạng và ảnh hỏng

Bước **clean**: mọi ảnh có cùng kích thước và cùng số kênh không, có ảnh nào không đọc được không. Tỉ lệ cao/rộng quyết định kích thước resize: biển số là hình chữ nhật ngang, ép về hình vuông sẽ làm méo chữ.

In [ ]:
def value_table(col):
    t = df[col].value_counts().rename("số ảnh").to_frame()
    t["tỉ lệ (%)"] = (t["số ảnh"] / len(df) * 100).round(2)
    return t

df["size"] = df["width"].astype(str) + "×" + df["height"].astype(str)
df["aspect"] = df["width"] / df["height"]
print(f"Số kích thước khác nhau: {df['size'].nunique()}")
print("Kích thước (rộng×cao) phổ biến nhất:"); display(value_table("size").head(10))
print("Thống kê rộng, cao, tỉ lệ rộng/cao:")
display(df[["width", "height", "aspect"]].describe().round(2).T)
print("Chế độ màu:");            display(value_table("mode"))
print("Định dạng file:");        display(value_table("format"))
print("Dung lượng mỗi ảnh (KB):"); display(df["kb"].describe().round(1).to_frame().T)

print(f"Ảnh không đọc được: {len(bad)}")
if bad:
    display(pd.DataFrame(bad))

RESULTS["n_distinct_sizes"] = int(df["size"].nunique())
RESULTS["image_sizes"] = df["size"].value_counts().head(10).to_dict()
RESULTS["size_stats"] = df[["width", "height", "aspect"]].describe().round(3).to_dict()
RESULTS["image_modes"] = df["mode"].value_counts().to_dict()
RESULTS["image_formats"] = df["format"].astype(str).value_counts().to_dict()
RESULTS["file_kb"] = df["kb"].describe().round(2).to_dict()
RESULTS["n_unreadable"] = len(bad)
RESULTS["unreadable"] = bad[:50]

> **Nhận xét**
>
> - Toàn bộ 8.721 ảnh có **cùng kích thước 224×128, cùng 3 kênh RGB, cùng định dạng JPEG**. Không có ảnh nào hỏng.
> - Tỉ lệ rộng/cao là 1,75, đúng dáng chữ nhật ngang của biển số.
> - Dung lượng mỗi ảnh 9–27 KB, trung bình 17 KB: ảnh đã được nén JPEG khá mạnh, nên các chi tiết nhỏ (dòng khẩu hiệu, tem năm) có thể bị nhoè.
>
> **Đề xuất:** không cần bước resize để đồng nhất dữ liệu. Giữ nguyên tỉ lệ 1,75 khi đưa vào mô hình: dùng **128×224 (cao×rộng)**, hoặc 64×112 để train nhanh hai mô hình tự thiết kế. Không ép về hình vuông 224×224 vì chữ sẽ bị bóp hẹp theo chiều ngang; các mạng pretrained có global average pooling vẫn nhận được ảnh chữ nhật.

## 5. Ảnh mẫu

Mỗi lớp một ảnh (lấy ngẫu nhiên), sau đó xem kỹ vài lớp để thấy độ đa dạng **bên trong** một lớp. Một bang thường có nhiều mẫu biển khác nhau (biển thường, biển kỷ niệm, biển theo năm), nên cùng một nhãn có thể trông rất khác nhau.

In [ ]:
def show_grid(items, ncols, title, name, cell=1.9, fontsize=7.5, lines=1):
    # items: list of (split, idx, caption)
    nrows = int(np.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(cell * ncols, (cell * ASPECT + 0.22 + 0.14 * lines) * nrows + 0.6))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (split, idx, cap) in zip(axes, items):
        ax.imshow(load_img(split, idx)); ax.set_title(cap, fontsize=fontsize, fontweight="normal",
                                                       loc="center", pad=3, color=C["text2"])
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold", color=C["text"])
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.45 / fig.get_figheight()))
    save(fig, name, photo=True)

data = df                                    # bộ đã gộp
one_per_class = (data.groupby("label", sort=False).sample(1, random_state=SEED)
                   .set_index("label").reindex(CLASS_NAMES).dropna())
items = [(r["split"], int(r["idx"]), lab) for lab, r in one_per_class.iterrows()]
show_grid(items, ncols=8, title="Mỗi lớp một ảnh mẫu (toàn bộ ảnh)", name="fig05a_one_per_class", cell=1.7)

In [ ]:
wanted = ["CALIFORNIA", "TEXAS", "NEW YORK", "FLORIDA", "ALASKA", "WYOMING"]
low = {c.lower(): c for c in CLASS_NAMES}
focus = [low[w.lower()] for w in wanted if w.lower() in low] or CLASS_NAMES[:6]
K = 8
items = []
for c in focus:
    sub = data[data["label"] == c]
    for _, r in sub.sample(min(K, len(sub)), random_state=SEED).iterrows():
        items.append((r["split"], int(r["idx"]), c))
show_grid(items, ncols=K, title="Độ đa dạng trong cùng một lớp (mỗi hàng một lớp)",
          name="fig05b_within_class", cell=1.7)

> **Nhận xét**
>
> - Ảnh được **cắt sát biển số, chụp chính diện, chữ đọc được**. Phần lớn biển có in tên bang, nhưng vị trí, kiểu chữ và kích cỡ của tên bang thay đổi rất nhiều; một số biển cổ chỉ ghi tên viết tắt.
> - **Độ đa dạng trong một lớp rất lớn.** Riêng California đã có biển nền trắng, nền vàng và nền xanh dương; Florida và Alaska có cả biển cổ (thập niên 1950–1970) lẫn biển hiện đại. Ngược lại, Wyoming gần như luôn có hình cao bồi cưỡi ngựa.
> - Dãy số trên biển không mang thông tin về bang, nên mô hình phải dựa vào **tên bang, bố cục và hoạ tiết nền**.
> - Có một số nhiễu thực tế: biển gỉ sét, khung biển che mép, watermark của trang ảnh, vài ảnh hơi nghiêng. Một số ảnh có vẻ là biển mẫu hoặc biển lưu niệm (cùng một dãy số in trên các nền khác nhau).
>
> **Kết luận:** đây là bài toán phân loại chi tiết (fine-grained) với nhiều "kiểu con" trong mỗi lớp. Mô hình cần đủ độ phân giải để thấy tên bang, nên không thu nhỏ ảnh quá mức; augment phải giữ được phần chữ.

## 6. Thống kê pixel theo kênh màu

`mean` và `std` từng kênh (thang 0–1), tính trên toàn bộ ảnh. Các giá trị này cho biết ảnh biển số khác ảnh tự nhiên (ImageNet) thế nào. Giá trị dùng thật để normalize sẽ được tính lại trên train sau khi chia; mô hình transfer learning dùng hàm `preprocess_input` của mạng pretrained.

In [ ]:
mean = ch_sum / max(n_px, 1)
std = np.sqrt(np.maximum(ch_sq / max(n_px, 1) - mean ** 2, 0))
norm = pd.DataFrame({"mean": mean.round(4), "std": std.round(4),
                     "ImageNet mean": [0.485, 0.456, 0.406], "ImageNet std": [0.229, 0.224, 0.225]},
                    index=["R", "G", "B"])
display(norm)
RESULTS["mean"] = [round(float(v), 4) for v in mean]
RESULTS["std"] = [round(float(v), 4) for v in std]

In [ ]:
h = hist256.sum(0)                       # (3, 256) trên toàn bộ ảnh
h = h / h.sum(1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(8.5, 3.6))
for c, (name, col, ls) in enumerate([("R", C["red"], "-"), ("G", C["green"], "--"), ("B", C["blue"], "-.")]):
    ax.plot(np.arange(256), h[c], color=col, linewidth=2, linestyle=ls, label=f"kênh {name}")
ax.set_xlim(0, 255); ax.set_ylim(0, None)
ax.set_xlabel("giá trị pixel (0–255)"); ax.set_ylabel("% số pixel")
ax.set_title("Phân phối giá trị pixel theo kênh màu (toàn bộ ảnh)", pad=30)
ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0.2, handlelength=2.6)
save(fig, "fig06_channel_histogram")

RESULTS["pixel_share_at_0_pct"] = [round(float(v), 3) for v in h[:, 0]]
RESULTS["pixel_share_at_255_pct"] = [round(float(v), 3) for v in h[:, 255]]
RESULTS["pixel_share_above_240_pct"] = [round(float(v), 2) for v in h[:, 241:].sum(1)]

> **Nhận xét**
>
> - Trên toàn bộ ảnh: mean = **[0,550; 0,541; 0,506]**, std = **[0,292; 0,273; 0,285]**. So với ImageNet, ảnh biển số **sáng hơn** và **phân tán hơn** (std khoảng 0,28 so với 0,225).
> - Histogram có **hai đỉnh**: vùng tối quanh 20–50 (chữ và các biển nền sẫm) và vùng sáng quanh 180–210 (nền biển). Đây là dạng điển hình của ảnh chữ trên nền.
> - Kênh B thấp hơn R và G ở vùng sáng, nên nhìn chung ảnh hơi ngả ấm (nền vàng, kem).
> - Khoảng 1% số pixel ở mỗi kênh nằm đúng giá trị 255 và khoảng 4–5% vượt 240: một phần ảnh bị cháy sáng ở nền trắng.
>
> **Đề xuất:** hai mô hình tự thiết kế nên normalize bằng mean / std của chính bộ dữ liệu, **tính lại trên train sau khi chia** (giá trị sẽ rất gần các số ở trên); mô hình transfer learning dùng hàm `preprocess_input` của chính mạng pretrained (mỗi mạng trong Keras có cách chuẩn hoá riêng) để khớp với trọng số đã huấn luyện sẵn.

## 7. Độ sáng và độ tương phản theo lớp

Mỗi hàng là một lớp: chấm là giá trị trung bình, vạch ngang là khoảng 10%–90% số ảnh của lớp đó. Ô màu bên trái là **màu trung bình** của lớp.

Với biển số, màu nền và màu chữ là một phần của thiết kế từng bang. Nếu các lớp tách nhau theo màu thì color jitter chỉ nên nhẹ và **không đổi hue**. Vạch 10%–90% dài nghĩa là trong cùng một bang có nhiều mẫu biển sáng tối khác nhau.

In [ ]:
def per_class(col):
    g = data.groupby("label")[col]
    t = pd.DataFrame({"mean": g.mean(), "median": g.median(), "p10": g.quantile(0.1), "p90": g.quantile(0.9)})
    return t.reindex(CLASS_NAMES).dropna()

mean_rgb = data.groupby("label")[["mean_r", "mean_g", "mean_b"]].mean().reindex(CLASS_NAMES).dropna()

def dot_range(col, title, xlabel, name, center="mean", log=False):
    t = per_class(col).sort_values(center)
    n = len(t)
    fig, (axs, ax) = plt.subplots(1, 2, figsize=(8.5, 0.24 * n + 1.3), sharey=True,
                                  gridspec_kw={"width_ratios": [0.035, 1], "wspace": 0.02})
    sw = (mean_rgb.loc[t.index].values / 255.0).reshape(n, 1, 3)
    axs.imshow(sw, aspect="auto", extent=(0, 1, n - 0.5, -0.5))
    axs.set_xticks([]); axs.set_yticks(range(n)); axs.set_yticklabels(t.index)
    axs.grid(False)
    for sp in axs.spines.values():
        sp.set_visible(False)
    y = np.arange(n)
    ax.hlines(y, t["p10"], t["p90"], color="#9ec5f4", linewidth=2.5)
    ax.scatter(t[center], y, s=38, color=C["blue"], edgecolor=C["surface"], linewidth=1.5, zorder=3)
    if log:
        ax.set_xscale("log")
    ax.set_ylim(-0.7, n - 0.3)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelleft=False)
    ax.set_xlabel(xlabel)
    axs.set_title(title)
    save(fig, name)
    return t

t_b = dot_range("brightness", "Độ sáng theo lớp (toàn bộ ảnh)",
                "độ sáng trung bình của ảnh (0–255) · chấm = trung bình, vạch = 10%–90%", "fig07a_brightness")
t_c = dot_range("contrast", "Độ tương phản theo lớp (toàn bộ ảnh)",
                "độ lệch chuẩn mức xám trong ảnh · chấm = trung bình, vạch = 10%–90%", "fig07b_contrast")

RESULTS["brightness_overall"] = data["brightness"].describe().round(2).to_dict()
RESULTS["contrast_overall"] = data["contrast"].describe().round(2).to_dict()
RESULTS["per_class_brightness"] = t_b.round(2).to_dict(orient="index")
RESULTS["per_class_contrast"] = t_c.round(2).to_dict(orient="index")
RESULTS["per_class_mean_rgb"] = mean_rgb.round(1).to_dict(orient="index")

> **Nhận xét**
>
> - Độ sáng trung bình của ảnh là 137,6 (thang 0–255), độ lệch chuẩn 31,7, trải từ 22 đến 236.
> - **Giữa các lớp khác nhau ít:** độ sáng trung bình theo lớp chỉ nằm trong khoảng 116–150, trừ DELAWARE (95) vì biển bang này chủ yếu nền xanh đen.
> - **Trong một lớp khác nhau nhiều:** khoảng 10%–90% của một lớp điển hình rộng khoảng 76 mức, lớn hơn nhiều so với chênh lệch giữa các lớp (khoảng 33 mức). Điều này khớp với việc mỗi bang có cả biển nền sáng lẫn nền tối.
> - Màu trung bình của hầu hết các lớp là xám-be. Ngoại lệ: DELAWARE (xám đen), VERMONT (xanh lá), NEW MEXICO (vàng), CONNECTICUT (xanh dương).
> - Độ tương phản trung bình 59; cao nhất là TEXAS (71, nhiều biển đen-trắng), thấp nhất là NEW MEXICO (48).
>
> **Đề xuất:** độ sáng không phải là tín hiệu phân loại, nên thay đổi **độ sáng và tương phản khoảng ±20%** khi augment là an toàn. Màu sắc thì có ích cho một số bang, nên **không đổi hue và không chuyển ảnh sang xám**.

## 8. Ảnh trung bình của mỗi lớp

Cộng tất cả ảnh của một lớp rồi chia trung bình.

- Nếu ảnh trung bình **còn thấy khung biển, dải màu hoặc vệt chữ** thì biển số được chụp canh giữa, cùng hướng → **không lật ảnh** (chữ sẽ bị ngược), chỉ xoay và dịch rất nhẹ, không crop mạnh.
- Lớp nào ảnh trung bình có màu và bố cục riêng rõ rệt thì dễ phân loại; lớp nào ra một khối xám nhạt giống nhau thì mô hình phải đọc được chi tiết nhỏ.

In [ ]:
ncols = 8
ids = [i for i in range(N_CLS) if mean_cnt[i] > 0]
nrows = int(np.ceil(len(ids) / ncols))
cw = 1.7
fig, axes = plt.subplots(nrows, ncols, figsize=(cw * ncols, (cw * MH / MW + 0.36) * nrows + 0.6))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")
center_bias, mean_img_std = {}, {}
for ax, i in zip(axes, ids):
    m = mean_img[i] / mean_cnt[i]
    ax.imshow(np.clip(m, 0, 255).astype(np.uint8), interpolation="nearest")
    ax.set_title(CLASS_NAMES[i], fontsize=7.5, fontweight="normal", loc="center", pad=3, color=C["text2"])
    g = m.mean(-1)
    qh, qw = MH // 4, MW // 4
    inner = g[qh:-qh, qw:-qw]
    border = (g.sum() - inner.sum()) / (g.size - inner.size)
    center_bias[CLASS_NAMES[i]] = round(float(inner.mean() - border), 2)
    mean_img_std[CLASS_NAMES[i]] = round(float(g.std()), 2)     # còn nhiều cấu trúc = std cao
fig.suptitle("Ảnh trung bình của mỗi lớp (toàn bộ ảnh)", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 1 - 0.45 / fig.get_figheight()))
save(fig, "fig08_mean_images", photo=True)

ms = pd.Series(mean_img_std).sort_values(ascending=False)
print("Độ lệch chuẩn mức xám của ảnh trung bình (cao = còn thấy rõ cấu trúc):")
display(pd.DataFrame({"cao nhất": ms.head(8).round(1).astype(str) + "  " + ms.head(8).index,
                      "thấp nhất": (ms.tail(8).round(1).astype(str) + "  " + ms.tail(8).index).values},
                     index=range(1, 9)))
RESULTS["center_minus_border_gray"] = center_bias
RESULTS["mean_image_gray_std"] = mean_img_std

> **Nhận xét**
>
> - Ảnh trung bình của **mọi lớp** đều còn thấy khung biển và một hàng ký tự mờ nằm ngang ở giữa. Vậy biển số trong bộ này luôn **canh giữa, thẳng và cùng tỉ lệ**.
> - Nhiều lớp còn giữ được thiết kế riêng: DELAWARE (nền xanh đen, chữ vàng), VERMONT (nền xanh lá), CONNECTICUT (nền xanh dương), NEW MEXICO (vàng), HAWAI (cầu vồng), FLORIDA (quả cam ở giữa), WYOMING (cao bồi), dải tên bang phía trên ở NEVADA, IDAHO, ARKANSAS, MINNESOTA.
> - Các lớp có ảnh trung bình rõ nhất là MISSOURI, DELAWARE, MINNESOTA; mờ nhất là NEW MEXICO, MONTANA, OHIO, OREGON, WASHINGTON DC. Ảnh trung bình mờ gợi ý bang đó có nhiều mẫu biển khác nhau, và có thể là các lớp khó hơn (cần kiểm chứng bằng kết quả mô hình).
>
> **Đề xuất cho augment:**
> - **Không lật ngang, không lật dọc:** chữ sẽ bị ngược và không còn là biển số thật.
> - Chỉ **xoay nhẹ (khoảng ±7–10°)**, dịch tối đa khoảng 6–8%, co giãn 0,9–1,1, có thể thêm biến dạng phối cảnh nhẹ.
> - **Không random crop mạnh:** tên bang thường nằm sát mép trên hoặc mép dưới, cắt mất là mất luôn tín hiệu chính.

## 9. Độ nét và các mẫu bất thường

Độ nét đo bằng phương sai của Laplacian: ảnh mờ hoặc gần như một màu cho giá trị rất thấp. Cần xem các ảnh **thấp bất thường so với chính lớp của nó**, và nhìn bằng mắt các ảnh ở hai đầu (mờ nhất, tối nhất, sáng nhất) để phát hiện ảnh không phải biển số hoặc ảnh chất lượng quá kém.

In [ ]:
t_s = dot_range("sharpness", "Độ nét theo lớp (toàn bộ ảnh)",
                "phương sai Laplacian, thang log · chấm = trung vị, vạch = 10%–90%",
                "fig09a_sharpness", center="median", log=True)
RESULTS["sharpness_overall"] = df["sharpness"].describe().round(2).to_dict()
RESULTS["per_class_sharpness"] = t_s.round(2).to_dict(orient="index")

In [ ]:
def extremes(col, n=12, largest=False):
    sub = df.nlargest(n, col) if largest else df.nsmallest(n, col)
    return [(r.split, int(r.idx), f"{r.label}\n{col}={getattr(r, col):.1f}") for r in sub.itertuples()], sub

it, sub_blur = extremes("sharpness")
show_grid(it, 6, "12 ảnh kém nét nhất toàn bộ dữ liệu", "fig09b_least_sharp", lines=2)
it, sub_flat = extremes("contrast")
show_grid(it, 6, "12 ảnh có độ tương phản thấp nhất", "fig09c_lowest_contrast", lines=2)
it, sub_dark = extremes("brightness")
show_grid(it, 6, "12 ảnh tối nhất", "fig09d_darkest", lines=2)
it, sub_bright = extremes("brightness", largest=True)
show_grid(it, 6, "12 ảnh sáng nhất", "fig09e_brightest", lines=2)

# ảnh lệch xa so với chính lớp của nó (z-score theo log độ nét)
z = df.assign(ls=np.log10(df["sharpness"] + 1e-3))
z["z"] = z.groupby("label")["ls"].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))
out_low = z[z["z"] < -3]
print(f"Ảnh mờ bất thường so với lớp của nó (z < -3): {len(out_low)} / {len(df):,}")
if len(out_low):
    display(out_low["label"].value_counts().head(10).rename("số ảnh").to_frame().T)
    sub = out_low.nsmallest(12, "z")
    show_grid([(r.split, int(r.idx), f"{r.label}\nz={r.z:.1f}") for r in sub.itertuples()], 6,
              "Ảnh mờ bất thường so với lớp của nó", "fig09f_blur_outliers", lines=2)

near_uniform = df[df["contrast"] < 5]
RESULTS["n_blur_outliers_z_lt_-3"] = int(len(out_low))
RESULTS["blur_outliers_by_class"] = out_low["label"].value_counts().head(15).to_dict()
RESULTS["blur_outlier_files"] = out_low.nsmallest(30, "z")["file"].tolist()
RESULTS["n_near_uniform_contrast_lt_5"] = int(len(near_uniform))
RESULTS["near_uniform_files"] = near_uniform["file"].head(30).tolist()
RESULTS["least_sharp_files"] = sub_blur["file"].tolist()
RESULTS["lowest_contrast_files"] = sub_flat["file"].tolist()
RESULTS["darkest_files"] = sub_dark["file"].tolist()
RESULTS["brightest_files"] = sub_bright["file"].tolist()

> **Nhận xét**
>
> - Độ nét (phương sai Laplacian) có trung vị 3.467, khoảng tứ phân vị 2.190–5.236, thấp nhất 144. **Không có ảnh nào gần như một màu.**
> - Theo lớp, thấp nhất là RHODE ISLAND, NEW JERSEY, MAINE; cao nhất là GEORGIA, TEXAS. Chênh lệch này đến từ kiểu thiết kế (biển trơn, ít chi tiết thì điểm thấp), không phải do ảnh hỏng.
> - Có 47 ảnh (0,5%) mờ bất thường so với lớp của mình. Xem trực tiếp 12 ảnh mờ nhất trong số đó thì **tất cả vẫn là biển số đọc được**: chủ yếu là ảnh độ phân giải thấp được phóng to, biển chữ nhạt trên nền nhạt, hoặc biển cổ bị gỉ.
> - Nhóm ảnh tối nhất phần lớn là các biển cổ nền đen; nhóm sáng nhất là ảnh bị cháy sáng, chữ nhạt trên nền trắng. Cả hai đều hợp lệ nhưng là **mẫu khó**.
> - Có ít nhất một ảnh bất thường về bố cục: một ảnh COLORADO chứa hai biển đặt cạnh nhau trên nền đen.
>
> **Đề xuất:** không loại ảnh nào vì lý do chất lượng. Các ảnh mờ và ảnh cháy sáng nên được giữ lại để mô hình gặp cả điều kiện xấu; có thể thêm Gaussian blur nhẹ vào augment.

## 10. Ảnh trùng lặp và rò rỉ giữa các split

Ảnh trùng gây hai vấn đề: (1) khi chia dữ liệu, nếu hai bản của cùng một ảnh rơi vào train và valid/test thì mô hình được "xem trước đáp án", kết quả đánh giá bị thổi phồng; (2) ảnh trùng trong train làm một mẫu được tính nhiều lần. Vì EDA đang xét bộ đã gộp, phần này tìm ảnh trùng **trên toàn bộ 3 thư mục**, đồng thời cho biết cặp nào vốn nằm ở hai thư mục khác nhau của bộ gốc.

Phần này **chỉ tìm và chỉ ra** các cặp ảnh giống nhau, không xoá gì. Cách tìm gồm ba bước:

1. **Trùng tuyệt đối:** hai ảnh có mọi pixel giống hệt nhau (so mã MD5 của pixel).
2. **So kỹ mọi cặp ảnh cùng bang:** trong mỗi bang, so từng ảnh với **mọi ảnh còn lại của bang đó** (tổng cộng khoảng 679 nghìn cặp). Với mỗi cặp, lấy vùng trung tâm của ảnh A ở độ phân giải 112×64 rồi so với ảnh B ở nhiều vị trí (dịch tối đa ±6 pixel dọc, ±8 pixel ngang) và nhiều mức co giãn (0,92–1,08). Tương quan lớn nhất trong các lần thử là **độ giống** của cặp. Bước này cần thiết vì bản sao thường bị cắt lệch hoặc lưu ở kích thước khác. Các phép so được viết thành phép nhân ma trận nên chạy hết trong khoảng 1–2 phút.
3. **Kiểm tra cặp khác bang:** số cặp khác bang lớn gấp khoảng 55 lần (khoảng 37 triệu cặp) nên không so kỹ hết; chỉ dùng hệ số tương quan của ảnh thu nhỏ 28×16 để tìm các cặp rất giống nhau mang hai nhãn khác nhau (dấu hiệu gán nhầm nhãn).

**Vì sao không lọc trước bằng ảnh thu nhỏ cho cặp cùng bang:** tương quan của ảnh thu nhỏ không chịu được lệch vị trí, nên nhiều bản sao bị cắt lệch có tương quan ảnh thu nhỏ thấp và sẽ bị loại trước khi kịp được so kỹ. Ô kết quả bên dưới đếm cụ thể có bao nhiêu cặp bị bỏ sót nếu chỉ so kỹ các cặp có tương quan ảnh thu nhỏ trên 0,80.

Cặp có độ giống **≥ 0,70** được coi là **cặp ảnh giống nhau**. Nhóm này gồm hai trường hợp: cùng một ảnh bị lưu lại, cắt lại hoặc đổi màu; và hai biển khác nhau cùng mẫu thiết kế (chỉ khác dãy số). Hai trường hợp này không tách được bằng một ngưỡng độ giống, nên EDA **không phân loại tiếp** mà đề xuất một cách xử lý chung cho cả hai: giữ mọi ảnh và xếp chung vào train. Như vậy không cần quyết định ảnh nào là bản sao để xoá.

Ngưỡng 0,70 được chọn bằng cách xem trực tiếp các cặp quanh ngưỡng (các hình bên dưới). Đặt ngưỡng hơi thấp là an toàn: ngưỡng chỉ quyết định ảnh nào phải đi chung vào train, không xoá ảnh nào, nên đặt thấp chỉ làm thêm vài ảnh vào train chứ không làm mất dữ liệu.


In [ ]:
# ---- bước 1: trùng tuyệt đối
exact = df[df.duplicated("md5", keep=False)].sort_values("md5")
g = exact.groupby("md5")
exact_info = pd.DataFrame({"số ảnh": g.size(), "số split": g["split"].nunique(), "số nhãn": g["label"].nunique()})
print(f"Nhóm ảnh trùng tuyệt đối: {len(exact_info)}")
if len(exact):
    display(exact[["md5", "split", "label", "key"]])

# ---- tương quan của ảnh thu nhỏ: dùng cho kiểm tra cặp khác bang (bước 3) và để so sánh với bước 2
T = np.stack(thumbs)
lab = df["label_id"].values
cand_same, cand_diff = [], []
for i in range(0, len(T), 1000):
    S = T[i:i + 1000] @ T.T
    a, b = np.where(S > 0.80)
    for x, y in zip(a, b):
        x, y = int(i + x), int(y)
        if x < y:
            (cand_same if lab[x] == lab[y] else cand_diff).append((x, y, float(S[x - i, y])))
print(f"Cặp có tương quan ảnh thu nhỏ > 0,80: {len(cand_same):,} cặp cùng bang, {len(cand_diff):,} cặp khác bang")

In [ ]:
# ---- bước 2: so kỹ mọi cặp ảnh cùng bang, có căn chỉnh
GROUP_T = 0.70      # từ ngưỡng này trở lên: hai ảnh được coi là giống nhau và phải nằm chung một tập

SCALES = (0.92, 0.96, 1.0, 1.04, 1.08)
KEEP_FROM = 0.40     # chỉ lưu các cặp từ mức này trở lên (để vẽ phân phối); thấp hơn là hai ảnh khác hẳn nhau

def unit_rows(v):
    # mỗi dòng: trừ trung bình, chia độ dài -> tích vô hướng của hai dòng chính là hệ số tương quan
    v = v.reshape(len(v), -1).astype(np.float32)
    v = v - v.mean(1, keepdims=True)
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-6)

def align_features(img):
    # A: vùng trung tâm 48x88 của ảnh 112x64; B: mọi phiên bản dịch / co giãn có cùng kích thước 48x88
    g = img.convert("L")
    center = np.asarray(g.resize((112, 64), PILImage.BILINEAR), dtype=np.float32)[8:-8, 12:-12]
    shifted = []
    for sc in SCALES:
        w, h = int(round(112 * sc)), int(round(64 * sc))
        B = np.asarray(g.resize((w, h), PILImage.BILINEAR), dtype=np.float32)
        for dy in range(-6, 7, 2):
            for dx in range(-8, 9, 2):
                y0, x0 = (h - 48) // 2 + dy, (w - 88) // 2 + dx
                if y0 >= 0 and x0 >= 0 and y0 + 48 <= h and x0 + 88 <= w:
                    shifted.append(B[y0:y0 + 48, x0:x0 + 88])
    return unit_rows(center[None])[0], unit_rows(np.stack(shifted))

recs, n_checked = [], 0
labels_arr = df["label"].values
for lab in tqdm(sorted(df["label"].unique()), desc="so kỹ từng bang"):
    idx = np.where(labels_arr == lab)[0]
    feats = [align_features(load_img(df["split"].iat[i], df["idx"].iat[i])) for i in idx]
    A = np.stack([f[0] for f in feats])                   # (n, 4224)
    V = feats[0][1].shape[0]                               # số phiên bản dịch / co giãn của mỗi ảnh
    S = np.empty((len(idx), len(idx)), dtype=np.float32)   # S[i, j] = độ giống của ảnh i với ảnh j
    for j0 in range(0, len(idx), 16):                      # 16 ảnh B một lần để không tốn bộ nhớ
        Bv = np.concatenate([f[1] for f in feats[j0:j0 + 16]])
        S[:, j0:j0 + 16] = (A @ Bv.T).reshape(len(idx), -1, V).max(-1)
    n_checked += len(idx) * (len(idx) - 1) // 2
    xs, ys = np.where(np.triu(S, 1) >= KEEP_FROM)          # mỗi cặp tính một lần
    for x, y in zip(xs, ys):
        a_, b_ = int(idx[x]), int(idx[y])
        recs.append(dict(a=a_, b=b_, image_a=df["key"].iat[a_], image_b=df["key"].iat[b_], label=lab,
                         split_a=df["split"].iat[a_], split_b=df["split"].iat[b_], similarity=round(float(S[x, y]), 4)))
pairs_all = pd.DataFrame(recs).sort_values("similarity", ascending=False).reset_index(drop=True)
print(f"Đã so {n_checked:,} cặp ảnh cùng bang · lưu {len(pairs_all):,} cặp có độ giống ≥ {KEEP_FROM}")

pairs_all["cross_split"] = pairs_all["split_a"] != pairs_all["split_b"]
dup_pairs = pairs_all[pairs_all["similarity"] >= GROUP_T].reset_index(drop=True)

# ---- gom các cặp thành nhóm (A giống B, B giống C -> A, B, C cùng một nhóm)
parent = list(range(len(df)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for r in dup_pairs.itertuples():
    ra, rb = find(r.a), find(r.b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)
grp = np.array([find(i) for i in range(len(df))])
gsize = pd.Series(grp).map(pd.Series(grp).value_counts()).values
in_group = gsize > 1
df["similar_group"] = np.where(in_group, grp, -1)       # -1: không giống ảnh nào

n_img, n_grp = int(in_group.sum()), len(set(grp[in_group]))
print(f"Cặp ảnh giống nhau (độ giống ≥ {GROUP_T}): {len(dup_pairs)} trong {n_checked:,} cặp cùng bang")
# so sánh: nếu chỉ so kỹ các cặp có tương quan ảnh thu nhỏ > 0,80 thì tìm được bao nhiêu cặp?
thumb_corr = np.array([float(T[r.a] @ T[r.b]) for r in dup_pairs.itertuples()])
n_via_thumb = int((thumb_corr > 0.80).sum())
print(f"Nếu lọc trước bằng ảnh thu nhỏ (> 0,80) thì chỉ tìm được {n_via_thumb} / {len(dup_pairs)} cặp này"
      f" · tương quan ảnh thu nhỏ thấp nhất trong các cặp: {thumb_corr.min():.2f}")
print(f"Số nhóm: {n_grp} (nhóm lớn nhất gồm {gsize.max()} ảnh) · số ảnh thuộc các nhóm: {n_img} ({n_img / len(df):.1%})")
print(f"Cặp nằm ở hai thư mục gốc khác nhau: {int(dup_pairs['cross_split'].sum())}")
print("Cặp giống nhau theo vị trí của hai ảnh:")
display(pd.crosstab(dup_pairs["split_a"], dup_pairs["split_b"]))
print("Ảnh thuộc các nhóm, theo thư mục gốc:", df.loc[in_group, "split"].value_counts().to_dict())
print("Bang có nhiều ảnh thuộc các nhóm nhất:")
display(df.loc[in_group, "label"].value_counts().head(10).rename("số ảnh").to_frame().T)

# ---- ghi danh sách cho bước tiền xử lý
dup_pairs[["image_a", "image_b", "label", "split_a", "split_b", "similarity"]].to_csv(
    os.path.join(OUT, "duplicate_pairs.csv"), index=False)
print("Đã ghi duplicate_pairs.csv vào", OUT)

RESULTS["duplicates"] = {
    "exact_md5_groups": int(len(exact_info)),
    "exact_md5_cross_split_groups": int((exact_info["số split"] > 1).sum()) if len(exact_info) else 0,
    "candidate_pairs_same_label": len(cand_same), "candidate_pairs_cross_label": len(cand_diff),
    "similar_pairs": int(len(dup_pairs)), "similar_pairs_cross_split": int(dup_pairs["cross_split"].sum()),
    "similar_groups": n_grp, "images_in_groups": n_img,
    "images_in_groups_by_split": df.loc[in_group, "split"].value_counts().to_dict(),
    "images_in_groups_by_label": df.loc[in_group, "label"].value_counts().to_dict(),
    "similar_pairs_by_split": pd.crosstab(dup_pairs["split_a"], dup_pairs["split_b"]).to_dict(),
    "group_threshold": GROUP_T, "method": "all_same_label_pairs", "same_label_pairs_checked": int(n_checked),
    "similar_pairs_found_by_thumbnail_filter": n_via_thumb,
}


In [ ]:
# phân phối độ giống của các cặp ảnh cùng bang (từ 0,40 trở lên), kèm ngưỡng 0,70; trục dọc thang log
fig, ax = plt.subplots(figsize=(8.5, 3.6))
bins = np.arange(0.40, 1.0001, 0.02)
ax.hist(pairs_all["similarity"].clip(0.40, 1.0), bins=bins, color=C["blue"], edgecolor=C["surface"], linewidth=1.5)
ax.set_yscale("log")
ymax = ax.get_ylim()[1]
for x, txt in [(GROUP_T, "0,70\ngiống nhau →")]:
    ax.axvline(x, color=C["text2"], linewidth=1)
    ax.text(x + 0.004, ymax * 0.8, txt, va="top", ha="left", fontsize=8.5, color=C["text"])
ax.set_xlim(0.40, 1.0)
ax.set_xlabel("độ giống sau khi căn chỉnh"); ax.set_ylabel("số cặp (thang log)")
ax.set_title(f"Độ giống của các cặp ảnh cùng bang ({len(pairs_all):,} cặp từ 0,40 trở lên)")
ax.grid(axis="x", visible=False)
save(fig, "fig10a_similarity_distribution")

In [ ]:
def show_pairs(p, title, name, n=9):
    items = []
    for r in p.head(n).itertuples():
        for i in (r.a, r.b):
            items.append((df["split"].iat[i], int(df["idx"].iat[i]),
                          f"{df['split'].iat[i]}/{df['label'].iat[i]}\n{os.path.basename(df['file'].iat[i])} · {r.similarity:.3f}"))
    if items:
        show_grid(items, 6, title, name, lines=2)

cross_pairs = dup_pairs[dup_pairs["cross_split"]]
show_pairs(cross_pairs, "Cặp ảnh giống nhau nằm ở HAI THƯ MỤC GỐC KHÁC NHAU (rò rỉ trong cách chia gốc) — mỗi cặp hai ô liền nhau",
           "fig10b_similar_cross_split", n=12)
same = dup_pairs[~dup_pairs["cross_split"]]
show_pairs(same.sample(min(9, len(same)), random_state=SEED).sort_values("similarity", ascending=False),
           "Cặp ảnh giống nhau trong cùng một thư mục gốc (lấy ngẫu nhiên 9 cặp)", "fig10c_similar_same_split")
show_pairs(dup_pairs.sort_values("similarity").head(6), "6 cặp có độ giống THẤP NHẤT vẫn được tính là giống nhau (ngay trên 0,70)",
           "fig10d_just_above_threshold", n=6)
band = pairs_all[(pairs_all["similarity"] >= GROUP_T) & (pairs_all["similarity"] < 0.75)]
show_pairs(band.head(6), "6 cặp có độ giống cao nhất trong khoảng 0,70–0,75",
           "fig10e_band_070_075", n=6)
show_pairs(pairs_all[pairs_all["similarity"] < GROUP_T].head(6), "6 cặp có độ giống CAO NHẤT dưới ngưỡng (ngay dưới 0,70)",
           "fig10f_just_below_threshold", n=6)


### Ảnh rất giống nhau nhưng mang hai nhãn khác nhau

Nếu cùng một ảnh xuất hiện dưới hai nhãn thì đó là nhãn nhiễu. Ô dưới liệt kê các cặp **khác bang** có tương quan ảnh thu nhỏ trên 0,90.

In [ ]:
cross = sorted([c for c in cand_diff if c[2] > 0.90], key=lambda c: -c[2])
print(f"Cặp khác bang có tương quan ảnh thu nhỏ > 0,90: {len(cross)}")
cross_df = pd.DataFrame([dict(a=a, b=b, image_a=df["key"].iat[a], image_b=df["key"].iat[b], similarity=round(s_, 3))
                         for a, b, s_ in cross])
if len(cross_df):
    display(cross_df[["image_a", "image_b", "similarity"]])
    show_pairs(cross_df, "Cặp ảnh khác bang giống nhau nhất", "fig10h_cross_label_lookalikes", n=6)
RESULTS["cross_label_lookalikes"] = cross_df[["image_a", "image_b", "similarity"]].values.tolist() if len(cross_df) else []

> **Nhận xét**
>
> - **Trùng tuyệt đối:** chỉ có 1 nhóm, và nó nằm ở hai thư mục khác nhau: `valid/OREGON/5.jpg` giống hệt `test/OREGON/1.jpg`.
> - **Cặp ảnh giống nhau:** trong 679.091 cặp ảnh cùng bang có **485 cặp có độ giống ≥ 0,70**. Chúng gom thành 286 nhóm (nhóm lớn nhất 18 ảnh), gồm tổng cộng **693 ảnh** (7,9% bộ dữ liệu): 638 ở `train`, 30 ở `valid`, 25 ở `test`. Bang có nhiều ảnh như vậy nhất là WYOMING (56 ảnh), rồi ARKANSAS (28).
> - **Lọc trước bằng ảnh thu nhỏ sẽ bỏ sót rất nhiều:** chỉ 242 trong 485 cặp có tương quan ảnh thu nhỏ trên 0,80; có cặp chỉ đạt −0,11. Ví dụ `NEW MEXICO/008` và `NEW MEXICO/101` (biển `NM 1912`, hai ảnh cắt khác nhau) có tương quan ảnh thu nhỏ −0,08 nhưng độ giống sau căn chỉnh 0,72, và đúng là cùng một tấm biển. Đây là lý do bước 2 so kỹ mọi cặp thay vì chỉ so các cặp đã lọc.
> - **Cách chia của bộ gốc có rò rỉ:** 58 trong 485 cặp nằm ở hai thư mục khác nhau (30 cặp `train`–`valid`, 26 cặp `train`–`test` và 2 cặp `valid`–`test`). Hình 10b cho thấy nhiều cặp trong số đó là cùng một tấm biển, tức là một số ảnh dùng để đánh giá đã có sẵn trong train.
> - **Trong nhóm giống nhau có cả hai trường hợp:** cùng một ảnh được lưu lại hoặc cắt lại (phần lớn các cặp có độ giống gần 1), và hai biển khác nhau cùng mẫu thiết kế. Hai trường hợp đan xen nhau ở cùng mức độ giống, nên không có ngưỡng nào tách được chúng một cách chắc chắn.
> - **Vì sao chọn 0,70:** các cặp ngay dưới 0,75 vẫn còn nhiều cặp là cùng một tấm biển (hình 10e: KANSAS `549 DNP`, MISSOURI `690 TW7`, ARKANSAS `D 1355`, TENNESSEE `999-QLM`), nên ngưỡng được hạ xuống 0,70. Quanh 0,70 (hình 10d, 10f) phần lớn đã là hai biển khác nhau cùng mẫu thiết kế. Tuy vậy không có ngưỡng nào tách sạch: xem hết 193 cặp trong khoảng 0,68–0,70 thì có khoảng 6 cặp vẫn là cùng một tấm biển (ví dụ KENTUCKY `527 CXS` ở 0,699, IDAHO `FS 259 HK`, MARYLAND `FXD 847`). Hạ ngưỡng xuống 0,68 sẽ phải dồn thêm khoảng 160 ảnh vào train chỉ để bắt thêm vài cặp, nên giữ 0,70; phần rò rỉ còn lại ước tính chỉ 1–2 ảnh test.
> - **Không thấy ảnh nào trùng mà mang hai nhãn khác nhau.** Hai cặp khác bang giống nhau nhất đều là ảnh khác nhau thật: một cặp là hai biển mẫu cùng dãy số `SWR 5149` nhưng in tên ARIZONA và INDIANA, cặp kia là hai biển hoàn toàn khác nhau.
>
> **Đề xuất cho bước tiền xử lý** (EDA không xoá gì, chỉ ghi danh sách ra `data/eda_outputs/duplicate_pairs.csv`):
> - **Không xoá ảnh nào.** Mọi ảnh thuộc một cặp giống nhau được gom theo nhóm và **xếp chung vào train** trước khi chia phần còn lại theo tỉ lệ. Như vậy không cặp nào bị tách sang hai tập, kể cả cặp trùng tuyệt đối ở Oregon.
> - Cái giá phải chấp nhận: train còn một số ảnh lặp lại, và valid/test hơi thiếu các mẫu biển có nhiều ảnh gần giống nhau (rõ nhất ở WYOMING).
>
> **Giới hạn:** cách tìm này bắt được trường hợp cùng một ảnh được lưu lại, cắt lại (trong phạm vi dịch và co giãn đã thử) hoặc đổi độ sáng. Hai bức ảnh khác nhau chụp cùng một tấm biển (khác góc, khác ánh sáng) có thể không bị phát hiện. Cặp khác bang chỉ được kiểm tra bằng ảnh thu nhỏ, nên có thể sót bản sao bị cắt lệch mang nhãn khác.


## 11. Độ giống nhau giữa các lớp (theo phân phối màu)

Mỗi lớp được tóm tắt bằng histogram màu (3 kênh × 32 khoảng, tính trên toàn bộ ảnh). Độ giống nhau của hai lớp là phần diện tích chung của hai histogram (0 = khác hẳn, 1 = giống hệt).

Đây là ước lượng **thô**: nó chỉ nhìn màu, không nhìn chữ và hoạ tiết. Hai bang có biển cùng tông màu (ví dụ nền trắng chữ xanh đậm) sẽ buộc mô hình phải học tên bang in trên biển và hoạ tiết nền để tách — đó là các cặp đáng theo dõi trong confusion matrix sau này.

In [ ]:
ok = [i for i in range(N_CLS) if hist256[i].sum() > 0]
names = [CLASS_NAMES[i] for i in ok]
h32 = hist256[ok].reshape(len(ok), 3, 32, 8).sum(-1)
h32 = h32 / h32.sum(-1, keepdims=True)                       # mỗi kênh chuẩn hoá về tổng 1
sim = np.minimum(h32[:, None], h32[None, :]).sum(-1).mean(-1)  # histogram intersection, trung bình 3 kênh

# sắp các lớp gần nhau đứng cạnh nhau (theo thành phần chính thứ nhất của ma trận)
w, v = np.linalg.eigh(sim - sim.mean(0, keepdims=True))
o = np.argsort(v[:, -1])
S = sim[np.ix_(o, o)].copy(); np.fill_diagonal(S, np.nan)

fig, ax = plt.subplots(figsize=(12, 11))
im = ax.imshow(S, cmap=SEQ, vmin=np.nanpercentile(S, 2), vmax=np.nanmax(S))
ax.set_xticks(range(len(o))); ax.set_yticks(range(len(o)))
ax.set_xticklabels([names[i] for i in o], rotation=90, fontsize=6.5)
ax.set_yticklabels([names[i] for i in o], fontsize=6.5)
ax.grid(False)
for sp in ax.spines.values():
    sp.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02); cb.outline.set_visible(False)
cb.set_label("độ giống nhau về màu (đậm hơn = giống hơn)")
ax.set_title("Độ giống nhau giữa các lớp theo phân phối màu (toàn bộ ảnh)")
save(fig, "fig11_class_similarity")

iu = np.triu_indices(len(ok), 1)
pairs = (pd.DataFrame({"lớp A": [names[i] for i in iu[0]], "lớp B": [names[j] for j in iu[1]],
                       "độ giống": sim[iu].round(4)})
           .sort_values("độ giống", ascending=False).reset_index(drop=True))
print("20 cặp lớp giống nhau nhất về màu:")
display(pairs.head(20))
print("10 cặp khác nhau nhất:")
display(pairs.tail(10))

nn = pd.Series({names[i]: float(np.nanmax(np.where(np.eye(len(ok))[i] == 1, np.nan, sim[i]))) for i in range(len(ok))})
print("Lớp tách biệt nhất về màu (độ giống với lớp gần nhất thấp):")
display(nn.sort_values().head(8).round(3).rename("độ giống với lớp gần nhất").to_frame().T)

RESULTS["most_similar_pairs"] = pairs.head(25).values.tolist()
RESULTS["least_similar_pairs"] = pairs.tail(10).values.tolist()
RESULTS["similarity_summary"] = pairs["độ giống"].describe().round(4).to_dict()
RESULTS["nearest_neighbour_similarity"] = nn.round(4).to_dict()

> **Nhận xét**
>
> - Phân phối màu của các lớp **rất giống nhau**: độ giống trung bình 0,86, trung vị 0,87, và 75% số cặp lớp có độ giống từ 0,83 trở lên.
> - Cặp giống nhau nhất là ALABAMA – GEORGIA (0,95); khác nhau nhất là DELAWARE – MISSIPPI (0,62).
> - Chỉ vài lớp tách biệt về màu: DELAWARE (độ giống với lớp gần nhất 0,84), rồi VERMONT và TEXAS.
> - Vậy **màu sắc chỉ là tín hiệu yếu**: một mô hình chỉ học màu sẽ nhận ra vài bang đặc biệt và nhầm lẫn phần còn lại. Điều này giải thích vì sao bài toán khó với CNN nông và vì sao transfer learning (đã học sẵn đặc trưng hình dạng, kết cấu) được kỳ vọng tốt hơn rõ rệt.
>
> **Các cặp nên theo dõi trong confusion matrix** (giống nhau nhất về màu): ALABAMA – GEORGIA, NEBRASKA – OKLAHOMA, IOWA – SOUTH DAKOTA, ALABAMA – TENNESSEE, NORTH DAKOTA – UTAH. Đây mới là giả thuyết từ màu sắc; cặp nhầm thật sự có thể khác vì mô hình còn nhìn chữ và hoạ tiết.

## 12. Tổng hợp và lưu kết quả

Ô dưới in các con số chính và ghi tất cả vào `data/eda_outputs/eda_results.json`. Cùng thư mục đó đã có danh sách ảnh trùng, bảng thống kê từng ảnh và mọi biểu đồ, dùng được cho báo cáo và slide.

In [ ]:
def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, float) and np.isnan(o):
        return None
    return o

with open(os.path.join(OUT, "eda_results.json"), "w", encoding="utf-8") as f:
    json.dump(_clean(RESULTS), f, ensure_ascii=False, indent=2)

print("TÓM TẮT")
print(f"- Số ảnh: {len(df):,} · số lớp: {N_CLS} · split: {RESULTS['splits']}")
print(f"- Ảnh mỗi lớp (bộ đã gộp): {counts['tổng'].min()} – {counts['tổng'].max()} (trung bình {counts['tổng'].mean():.1f})")
for s in SPLITS:
    print(f"- Ảnh mỗi lớp trong thư mục gốc {s}: {counts[s].min()} – {counts[s].max()} (trung bình {counts[s].mean():.1f})")
print(f"- Kích thước phổ biến: {df['size'].value_counts().index[0]} ({df['size'].value_counts().iloc[0] / len(df):.1%})"
      f" · số kích thước khác nhau: {df['size'].nunique()}")
print(f"- Chế độ màu: {RESULTS['image_modes']} · định dạng: {RESULTS['image_formats']}")
print(f"- Ảnh không đọc được: {len(bad)}")
print(f"- mean (toàn bộ ảnh): {RESULTS['mean']} · std (toàn bộ ảnh): {RESULTS['std']}")
DU = RESULTS["duplicates"]
print(f"- Tên lớp không khớp tên chuẩn (ở cả thư mục và plates.csv): {[o['tên thư mục'] for o in RESULTS['odd_class_names']]}")
print(f"- Cặp ảnh giống nhau (≥ {DU['group_threshold']}): {DU['similar_pairs']} (trong đó {DU['similar_pairs_cross_split']} cặp nằm ở hai thư mục gốc khác nhau)"
      f" · {DU['similar_groups']} nhóm, {DU['images_in_groups']} ảnh")
print(f"- Trùng tuyệt đối (MD5): {DU['exact_md5_groups']} nhóm")
print(f"- Ảnh mờ bất thường: {RESULTS['n_blur_outliers_z_lt_-3']} · ảnh gần như một màu: {RESULTS['n_near_uniform_contrast_lt_5']}")
print(f"- Cặp lớp giống nhau nhất về màu: {pairs.iloc[0, 0]} – {pairs.iloc[0, 1]} ({pairs.iloc[0, 2]:.3f})")

print("\nĐã lưu kết quả vào:", OUT)

## Kết luận: các việc cần làm ở bước tiền xử lý

EDA này không thay đổi dữ liệu. Bảng dưới tóm tắt những gì đã tìm thấy và việc tương ứng cần làm ở bước tiền xử lý.

| Vấn đề | Bằng chứng từ EDA | Việc cần làm ở bước tiền xử lý |
|---|---|---|
| **Cách chia của bộ gốc** | `valid` và `test` chỉ có 5 ảnh mỗi lớp (280 ảnh, sai số accuracy khoảng ±5,6 điểm). | Gộp cả ba thư mục thành một bảng chung, rồi chia lại khoảng 70 / 15 / 15, stratified theo lớp, cố định seed. |
| **Ảnh trùng / ảnh giống nhau** | So mọi cặp ảnh cùng bang: 485 cặp có độ giống ≥ 0,70 (693 ảnh, 286 nhóm); 58 cặp nằm ở hai thư mục khác nhau của bộ gốc; 1 cặp giống hệt từng pixel. | Không xoá ảnh nào. Gom các cặp thành nhóm và xếp mọi ảnh trong nhóm vào train, làm trước khi chia phần còn lại. Dùng `data/eda_outputs/duplicate_pairs.csv`. |
| **Tên lớp sai chính tả** | `HAWAI` (154 ảnh) và `MISSIPPI` (158 ảnh) sai ở tên thư mục của cả ba split, ở cột `labels` và cột `filepaths` của `plates.csv`. | Sau khi gộp, sửa một lần trên cột nhãn của bảng chung thành `HAWAII` và `MISSISSIPPI`. Thư mục và `plates.csv` của bộ mới được ghi ra theo nhãn đã sửa nên tự khớp nhau; bộ gốc giữ nguyên. |
| **Mất cân bằng lớp** | 96–185 ảnh mỗi lớp, tỉ lệ max/min 1,9 (do CNMI). | Không cần cân bằng lại. Báo cáo accuracy và F1 macro. |
| **Chất lượng ảnh** | 0 ảnh hỏng, 0 ảnh gần như một màu, mọi ảnh 224×128 RGB JPEG. | Không loại ảnh nào vì chất lượng. Không cần resize để đồng nhất. |
| **Kích thước đầu vào** | Tỉ lệ rộng/cao 1,75. | Giữ tỉ lệ: 128×224 (cao×rộng), hoặc 64×112 để train nhanh. |
| **Normalize** | mean [0,550; 0,541; 0,506], std [0,292; 0,273; 0,285] trên toàn bộ ảnh. | Tính lại trên train sau khi chia, dùng cho hai CNN tự thiết kế; transfer learning dùng `preprocess_input` của mạng pretrained. |
| **Augment nên dùng** | Biển luôn canh giữa và thẳng; độ sáng thay đổi nhiều trong cùng một lớp. | Xoay ±7–10°, dịch ≤ 6–8%, co giãn 0,9–1,1, phối cảnh nhẹ, độ sáng và tương phản ±20%, blur nhẹ. |
| **Augment không dùng** | Chữ có hướng; tên bang nằm sát mép; màu là đặc trưng của một số bang. | Không lật ngang/dọc, không random crop mạnh, không đổi hue, không chuyển xám. |

**Thứ tự thực hiện ở bước tiền xử lý**

Bảng trên xếp theo vấn đề. Khi làm thì theo thứ tự sau:

1. **Gộp:** đọc ảnh của `train`, `valid`, `test` vào một bảng chung, giữ lại đường dẫn gốc của từng ảnh.
2. **Đánh dấu ảnh giống nhau:** gom các cặp có độ giống ≥ 0,70 thành nhóm (không xoá ảnh nào).
3. **Sửa nhãn:** thay `HAWAI` và `MISSIPPI` trên cột nhãn của bảng chung.
4. **Chia lại:** ảnh thuộc các nhóm ở bước 2 vào thẳng train; phần còn lại chia 70 / 15 / 15 theo từng lớp, seed cố định.
5. **Ghi bộ mới:** sao chép ảnh sang thư mục mới theo split và nhãn đã sửa, kèm `plates.csv` mới.
6. **Normalize và augment:** tính mean / std trên train mới, định nghĩa augment cho train.

Gộp trước rồi mới sửa nhãn vì khi đó mỗi tên lớp chỉ phải sửa ở một chỗ (một cột của bảng chung), thay vì ba thư mục và hai cột CSV của bộ gốc. Bước 2 phải đứng trước bước 4, để hai ảnh giống nhau không rơi vào hai tập khác nhau; thứ tự giữa bước 2 và bước 3 không ảnh hưởng tới kết quả.

**Về độ khó của bài toán:** 56 lớp, khoảng 156 ảnh mỗi lớp trước khi chia, mỗi bang có nhiều mẫu biển khác nhau và màu sắc chỉ là tín hiệu yếu. Có thể kỳ vọng khoảng cách rõ giữa CNN đơn giản, CNN nhiều block và transfer learning.

**Giới hạn của phần EDA này**

- Độ giống nhau giữa các lớp chỉ đo bằng màu, chưa đo bằng hình dạng hay nội dung chữ.
- Bước tìm ảnh trùng có thể bỏ sót những ảnh khác nhau chụp cùng một tấm biển; cặp khác bang chỉ được kiểm tra bằng ảnh thu nhỏ.
- Chưa kiểm tra từng nhãn; EDA không phát hiện ảnh nào trùng mà mang hai nhãn khác nhau, nhưng điều đó chưa đủ để khẳng định không có ảnh gán nhầm bang.